# Exploring the unusual-plot model

A short, runnable tour: the numbers we give the model, how odd plots score, and what a few of them look like.
It builds its own small dataset in memory (no files, no database), so it runs top to bottom.

**Honest caveat:** the plots are *generated* by `db/seed.py`. A model trained on generated data mostly relearns the generator.
Nothing here shows it would work on real land records.

In [ ]:
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "backend").is_dir():          # find the repo root whether this runs from ml/ or from the root
    root = root.parent
sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
from ml import features, make_dataset, train_unusual, explain
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})

## 1. A training set: 2,000 generated plots, about 8% with a planted problem

In [ ]:
data = make_dataset.seed.generate(2000, 0.08, 2026)
plots, zones, labels = make_dataset.inputs_from_generated(data)
ulpins, matrix = features.features_for_all(plots, zones)
X = np.array(matrix)
has_problem = np.array([bool(labels.get(u)) for u in ulpins])
print(len(ulpins), "plots,", int(has_problem.sum()), "with a planted problem")
print("features:", ", ".join(features.FEATURE_NAMES))

## 2. What the features look like
Most plots are alike. The few that are not are what the forest picks out.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 5.5))
for ax, name in zip(axes.flat, ["area_sqm", "compactness", "aspect_ratio", "recorded_over_mapped",
                                 "years_since_tax_paid", "neighbour_count"]):
    values = X[:, features.FEATURE_NAMES.index(name)]
    ax.hist(values, bins=40, color="#8fa3c4")
    ax.set_title(name, fontsize=10)
fig.tight_layout(); plt.show()

## 3. Train (no labels) and score
An Isolation Forest cuts the data with random lines. A plot that is **alone** after a few cuts is easy to isolate, so it scores high ("unusual").
A plot lost in the crowd needs many cuts, so it scores low.

In [ ]:
bundle = train_unusual.fit(X, seed=2026)
scores = train_unusual.score(bundle, X)
flagged = scores >= bundle["threshold"]
print(f"threshold (95th percentile of the training scores): {bundle['threshold']:.3f}")
print(f"flagged as unusual: {int(flagged.sum())} of {len(X)}")

plt.figure(figsize=(8, 3.5))
plt.hist(scores[~has_problem], bins=50, alpha=.8, label="no planted problem", color="#8fa3c4")
plt.hist(scores[has_problem], bins=50, alpha=.8, label="has a planted problem", color="#e5535f")
plt.axvline(bundle["threshold"], color="black", linestyle="--", label="unusual if to the right")
plt.xlabel("unusualness score (higher = more unusual)"); plt.ylabel("plots"); plt.legend(); plt.show()

## 4. Did the unusual set find the planted problems? (labels used only now)

In [ ]:
tp = int((flagged & has_problem).sum()); fp = int((flagged & ~has_problem).sum())
print(f"flagged plots with a planted problem: {tp} of {int(flagged.sum())}  (precision {tp / flagged.sum():.0%}; "
      f"luck would give {has_problem.mean():.0%})")
print(f"problem plots caught: {tp} of {int(has_problem.sum())}  (recall {tp / has_problem.sum():.0%}; flagging at random would catch about 5%)")

## 5. What do the most unusual plots look like, and why?
Each title shows the plot's id and its top reason (SHAP when installed, otherwise z-scores).

In [ ]:
from shapely.affinity import scale
explainer = explain.Explainer(bundle)
by_id = {p["ulpin"]: p for p in plots}
top = np.argsort(-scores)[:4]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, i in zip(axes, top):
    p = by_id[ulpins[i]]
    cx, cy = p["geom"].centroid.x, p["geom"].centroid.y
    shown = [q for q in plots if q["geom"].centroid.distance(p["geom"].centroid) < 0.0012]
    for q in shown:
        g = features.to_metres(q["geom"]) if q["geom"].is_valid else features.to_metres(q["geom"]).convex_hull
        xs, ys = g.exterior.xy if g.geom_type == "Polygon" else ([], [])
        ax.fill(xs, ys, alpha=.25, color="#e5535f" if q is p else "#8fa3c4")
        ax.plot(xs, ys, color="#0a1429", linewidth=1)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"{ulpins[i][-6:]}  score {scores[i]:.2f}", fontsize=9)
    ax.set_xlabel(explainer.reasons(matrix[i], 1)[0][:42] + "...", fontsize=7)
fig.tight_layout(); plt.show()
for i in top:
    print(ulpins[i], "- planted:", labels.get(ulpins[i]) or "nothing")
    for line in explainer.reasons(matrix[i]):
        print("   -", line)

## What to take away
* The model only says a plot is **rare**, not that it is **wrong**.
* It cannot see names, survey numbers or missing records, because those are not in the feature list.
* It adds **no points** and never changes a band. The score still comes only from the eight checks in `backend/app/checks.py`.